In [ ]:
import time
import torch
from genkit.diffusion import DDPMV, DLPMEps
from genkit.flow_matching import GaussianFlowLinear
from genkit.training import train
from _constants import ALPHA, DEVICE, DIM, N_STEPS, N_TEST, N_TRIALS, TRAIN_KWARGS
from _guidance import Guidance, make_guidance_dataset
from _utils import add_test_vs_true_sample, evaluate_samples, load_alpha_stable_data, make_net, plot_tail_results


In [ ]:
x_train, x_test = load_alpha_stable_data()

DDPM_SIGMA_MAX = 5.0
GUIDANCE_BATCH_SIZE = 128
GUIDANCE_N_SAMPLES = 10_000
GUIDANCE_N_ROLLOUTS = 16
GUIDANCE_EPOCHS = 128
GUIDANCE_LR = 5e-4
GUIDANCE_TEMPERATURE = 5.0
GUIDANCE_STRENGTHS = (10.0, )
MAX_ABS_REWARD = 5.0

train_radius = x_train.reshape(x_train.shape[0], -1).norm(dim=1).sort().values.to(DEVICE)


def make_tail_ratio_reward(base_samples):
    base_radius = base_samples.detach().reshape(base_samples.shape[0], -1).norm(dim=1).sort().values.to(DEVICE)
    eps = 0.5 / min(len(train_radius), len(base_radius))

    def empirical_survival(sorted_radius, radius):
        sorted_radius = sorted_radius.to(device=radius.device, dtype=radius.dtype)
        index = torch.searchsorted(sorted_radius, radius.contiguous(), right=False)
        return ((len(sorted_radius) - index).to(radius.dtype) / len(sorted_radius)).clamp_min(eps)

    def reward(x):
        radius = x.reshape(x.shape[0], -1).norm(dim=1)
        target_survival = empirical_survival(train_radius, radius)
        base_survival = empirical_survival(base_radius, radius)
        return (target_survival.log() - base_survival.log()).clamp(-MAX_ABS_REWARD, MAX_ABS_REWARD)

    return reward


In [ ]:
rows = []
for trial in range(1, N_TRIALS + 1):
    torch.manual_seed(trial - 1)
    print(f"Trial {trial} / {N_TRIALS}: ", end="")

    start_time = time.time()
    ddpm = DDPMV(
        net=make_net(DIM),
        dim=DIM,
        n_steps=N_STEPS,
        sigma_max=DDPM_SIGMA_MAX,
        sampler="ddpm",
        device=DEVICE,
    )
    ddpm, _ = train(ddpm, x_train, **TRAIN_KWARGS)
    ddpm_samples = ddpm.sample(N_TEST)
    rows.append(evaluate_samples("DDPM", trial, x_test, ddpm_samples))
    print(f"'DDPM' done in {time.time() - start_time:.2f} s", end="", flush=True)

    start_time = time.time()
    dlpm = DLPMEps(
        net=make_net(DIM),
        alpha=ALPHA,
        dim=DIM,
        n_steps=N_STEPS,
        n_trial_A=1,
        n_trial_G=1,
        reduce_type="mean",
        device=DEVICE,
    )
    dlpm, _ = train(dlpm, x_train, **TRAIN_KWARGS)
    rows.append(evaluate_samples("DLPM", trial, x_test, dlpm.sample(N_TEST)))
    print(f", 'DLPM' done in {time.time() - start_time:.2f} s", end="", flush=True)

    start_time = time.time()
    gfl = GaussianFlowLinear(
        net=make_net(DIM),
        dim=DIM,
        n_steps=N_STEPS,
        sample_steps=N_STEPS,
        device=DEVICE,
    )
    gfl, _ = train(gfl, x_train, **TRAIN_KWARGS)
    rows.append(evaluate_samples("Gaussian Linear Flow", trial, x_test, gfl.sample(N_TEST)))
    print(f", 'Gaussian Linear Flow' done in {time.time() - start_time:.2f} s", end="", flush=True)

    start_time = time.time()
    guidance = Guidance(ddpm, make_net(DIM, output_dim=1))
    guidance_dataset = make_guidance_dataset(
        ddpm,
        make_tail_ratio_reward(ddpm_samples),
        temperature=GUIDANCE_TEMPERATURE,
        n_samples=GUIDANCE_N_SAMPLES,
        n_rollouts=GUIDANCE_N_ROLLOUTS,
        batch_size=GUIDANCE_BATCH_SIZE,
    )
    guidance.fit(
        guidance_dataset,
        epochs=GUIDANCE_EPOCHS,
        batch_size=GUIDANCE_BATCH_SIZE,
        lr=GUIDANCE_LR,
    )
    print(f", guidance fit done in {time.time() - start_time:.2f} s", end="", flush=True)

    for strength in GUIDANCE_STRENGTHS:
        start_time = time.time()
        name = f"DDPM + guidance (strength={strength:g})"
        guided_samples = guidance.sample(N_TEST, strength=strength)
        rows.append(evaluate_samples(name, trial, x_test, guided_samples))
        print(f", '{name}' done in {time.time() - start_time:.2f} s", end="", flush=True)

    print(" .", flush=True)

add_test_vs_true_sample(rows, x_test)


In [ ]:
fig, ax = plot_tail_results(rows, show_std=False)
fig.show()